In [1]:
!pip install gradio


In [ ]:
"""
EduLight-64: Educational Lightweight Block Cipher
-------------------------------------------------
Block size : 64 bits (8 bytes)
Key size   : 128 bits (16 bytes)
Structure  : Substitution-Permutation Network (SPN)
Rounds     : default 16

 Educational / academic use only.
Do NOT use this cipher to protect real data.
"""

from typing import List
from collections import Counter
import secrets

# -----------------------------
# 1. Core S-box and permutation
# -----------------------------

# 4-bit S-box (permutation of 0..15)
SBOX = [0xC, 0x5, 0x6, 0xB,
        0x9, 0x0, 0xA, 0xD,
        0x3, 0xE, 0xF, 0x8,
        0x4, 0x7, 0x1, 0x2]

# Inverse S-box
INV_SBOX = [0] * 16
for i, v in enumerate(SBOX):
    INV_SBOX[v] = i

# Bit permutation (PRESENT-style)
P = [0] * 64
for i in range(63):
    P[i] = (16 * i) % 63
P[63] = 63

# Inverse permutation
IP = [0] * 64
for i, v in enumerate(P):
    IP[v] = i

# -----------------------------
# 2. Round functions
# -----------------------------

def sbox_layer(state: int) -> int:
    """Apply S-box on each 4-bit nibble of the 64-bit state."""
    out = 0
    for i in range(16):  # 16 nibbles
        nib = (state >> (4 * i)) & 0xF
        sn = SBOX[nib]
        out |= (sn << (4 * i))
    return out

def inv_sbox_layer(state: int) -> int:
    """Inverse S-box layer."""
    out = 0
    for i in range(16):
        nib = (state >> (4 * i)) & 0xF
        sn = INV_SBOX[nib]
        out |= (sn << (4 * i))
    return out

def perm_layer(state: int) -> int:
    """Bit permutation layer."""
    out = 0
    for i in range(64):
        bit = (state >> i) & 1
        if bit:
            out |= (1 << P[i])
    return out

def inv_perm_layer(state: int) -> int:
    """Inverse bit permutation layer."""
    out = 0
    for i in range(64):
        bit = (state >> i) & 1
        if bit:
            out |= (1 << IP[i])
    return out

# -----------------------------
# 3. Key schedule
# -----------------------------

def expand_key(master_key: bytes, rounds: int = 16) -> List[int]:
    """
    Generate (rounds + 1) round keys, each 64 bits, from 128-bit master key.
    """
    assert len(master_key) == 16  # 128 bits
    k = int.from_bytes(master_key, "big")  # 128-bit integer
    round_keys = []

    for r in range(rounds + 1):
        # Use the most significant 64 bits as round key
        round_keys.append((k >> 64) & ((1 << 64) - 1))

        # 1) Rotate left by 13 bits
        k = ((k << 13) & ((1 << 128) - 1)) | (k >> (128 - 13))

        # 2) Apply S-box to the leftmost 2 nibbles (top 8 bits)
        left16 = (k >> 112) & 0xFFFF
        n0 = (left16 >> 12) & 0xF
        n1 = (left16 >> 8) & 0xF
        n0 = SBOX[n0]
        n1 = SBOX[n1]
        left16_new = (n0 << 12) | (n1 << 8) | (left16 & 0xFF)
        k = (k & ((1 << 112) - 1)) | (left16_new << 112)

        # 3) XOR round counter into key (simple diffusion of round index)
        k ^= (r & 0x1F) << 59  # inject 5 bits of round counter

    return round_keys

# -----------------------------
# 4. Block encryption/decryption
# -----------------------------

def encrypt_block(block: bytes, round_keys: List[int]) -> bytes:
    """Encrypt a single 64-bit block."""
    assert len(block) == 8
    R = len(round_keys) - 1
    state = int.from_bytes(block, "big")

    # Initial AddRoundKey
    state ^= round_keys[0]

    # Main rounds
    for r in range(1, R):
        state = sbox_layer(state)
        state = perm_layer(state)
        state ^= round_keys[r]

    # Final round: S-box + AddRoundKey (no permutation)
    state = sbox_layer(state)
    state ^= round_keys[R]

    return state.to_bytes(8, "big")

def decrypt_block(block: bytes, round_keys: List[int]) -> bytes:
    """Decrypt a single 64-bit block."""
    assert len(block) == 8
    R = len(round_keys) - 1
    state = int.from_bytes(block, "big")

    # Inverse final round
    state ^= round_keys[R]
    state = inv_sbox_layer(state)

    # Inverse main rounds
    for r in range(R - 1, 0, -1):
        state ^= round_keys[r]
        state = inv_perm_layer(state)
        state = inv_sbox_layer(state)

    # Inverse initial AddRoundKey
    state ^= round_keys[0]

    return state.to_bytes(8, "big")

# Quick self-test to be sure encrypt/decrypt are consistent
def _self_test_roundtrip():
    for rounds in [12, 16, 20]:
        for _ in range(50):
            key = secrets.token_bytes(16)
            rk = expand_key(key, rounds=rounds)
            pt = secrets.token_bytes(8)
            ct = encrypt_block(pt, rk)
            dt = decrypt_block(ct, rk)
            assert dt == pt
    print("Block cipher roundtrip OK")

_self_test_roundtrip()

# -----------------------------
# 5. CTR mode for variable-length data
# -----------------------------

def encrypt_ctr(data: bytes, key: bytes, nonce: bytes, rounds: int = 16) -> bytes:
    """
    Encrypt arbitrary-length data using EduLight-64 in CTR mode.
    key  : 16 bytes
    nonce: 8 bytes (initial counter)
    """
    assert len(key) == 16
    assert len(nonce) == 8

    round_keys = expand_key(key, rounds=rounds)
    out = bytearray()
    nonce_int = int.from_bytes(nonce, "big")

    for i in range(0, len(data), 8):
        block = data[i:i+8]

        counter = (nonce_int + (i // 8)) & ((1 << 64) - 1)
        counter_block = counter.to_bytes(8, "big")

        keystream = encrypt_block(counter_block, round_keys)
        keystream_slice = keystream[:len(block)]

        cipher_block = bytes(b ^ k for b, k in zip(block, keystream_slice))
        out.extend(cipher_block)

    return bytes(out)

def decrypt_ctr(cipher: bytes, key: bytes, nonce: bytes, rounds: int = 16) -> bytes:
    """Decrypt data in CTR mode (same operation as encryption)."""
    return encrypt_ctr(cipher, key, nonce, rounds=rounds)

# Quick CTR test
def _self_test_ctr():
    for _ in range(20):
        key = secrets.token_bytes(16)
        nonce = secrets.token_bytes(8)
        data = secrets.token_bytes(secrets.randbelow(100) + 1)
        c = encrypt_ctr(data, key, nonce)
        d = decrypt_ctr(c, key, nonce)
        assert d == data
    print("CTR mode roundtrip OK")

_self_test_ctr()


Block cipher roundtrip OK
CTR mode roundtrip OK


In [3]:
# -----------------------------
# 6. Helper functions for tests
# -----------------------------

def flip_bit(data: bytes, bit_index: int) -> bytes:
    """
    Flip a single bit in 'data'.
    bit_index: 0 = MSB of first byte, 63 = LSB of last byte (for 8-byte blocks).
    """
    byte_index = bit_index // 8
    bit_in_byte = 7 - (bit_index % 8)
    mask = 1 << bit_in_byte
    ba = bytearray(data)
    ba[byte_index] ^= mask
    return bytes(ba)

def hamming_distance_bytes(a: bytes, b: bytes) -> int:
    """Bit-level Hamming distance between two equal-length byte strings."""
    assert len(a) == len(b)
    dist = 0
    for x, y in zip(a, b):
        v = x ^ y
        dist += v.bit_count()
    return dist

# -----------------------------
# 7. Avalanche tests
# -----------------------------

def avalanche_plaintext(rounds: int = 16, trials: int = 200) -> float:
    """
    Flip 1 bit in plaintext, measure average bits changed in ciphertext.
    Ideal for 64-bit block is ~32 bits on average.
    """
    key = secrets.token_bytes(16)
    round_keys = expand_key(key, rounds=rounds)
    total = 0

    for _ in range(trials):
        pt = secrets.token_bytes(8)
        ct1 = encrypt_block(pt, round_keys)

        bit = secrets.randbelow(64)
        pt2 = flip_bit(pt, bit)
        ct2 = encrypt_block(pt2, round_keys)

        total += hamming_distance_bytes(ct1, ct2)

    avg = total / trials
    return avg

def avalanche_key(rounds: int = 16, trials: int = 200) -> float:
    """
    Flip 1 bit in key, measure average bits changed in ciphertext.
    """
    total = 0

    for _ in range(trials):
        key = secrets.token_bytes(16)
        data = secrets.token_bytes(8)

        rk1 = expand_key(key, rounds=rounds)
        ct1 = encrypt_block(data, rk1)

        bit = secrets.randbelow(128)
        key_int = int.from_bytes(key, "big")
        key_int ^= (1 << bit)
        key2 = key_int.to_bytes(16, "big")

        rk2 = expand_key(key2, rounds=rounds)
        ct2 = encrypt_block(data, rk2)

        total += hamming_distance_bytes(ct1, ct2)

    avg = total / trials
    return avg

# -----------------------------
# 8. Simple frequency test on ciphertext bytes
# -----------------------------

def frequency_test(num_blocks: int = 5000, rounds: int = 16) -> Counter:
    """
    Encrypt a sequence of counters and return byte frequency distribution
    of ciphertext bytes.
    """
    key = secrets.token_bytes(16)
    nonce = secrets.token_bytes(8)
    round_keys = expand_key(key, rounds=rounds)

    counts = Counter()

    base = int.from_bytes(nonce, "big")
    for i in range(num_blocks):
        block = (base + i).to_bytes(8, "big")
        ct = encrypt_block(block, round_keys)
        for b in ct:
            counts[b] += 1

    return counts

# Optional: one-time demo
print("Average bit flips (plaintext avalanche):", avalanche_plaintext())
print("Average bit flips (key avalanche)      :", avalanche_key())

freq = frequency_test(1000)
print("Distinct byte values seen:", len(freq))


Average bit flips (plaintext avalanche): 32.06
Average bit flips (key avalanche)      : 31.96
Distinct byte values seen: 256


In [4]:
import gradio as gr
import hashlib

# ------------- Small utilities ------------- #

def _hex_normalize(s: str, expected_len: int) -> bytes:
    """
    Normalize a hex string to fixed length in bytes.
    - Removes spaces
    - Adds leading 0 if odd length
    - Pads with zeros or truncates to expected_len bytes
    """
    s = s.strip().replace(" ", "").lower()
    if len(s) == 0:
        raise ValueError("Value cannot be empty")
    if len(s) % 2 == 1:
        s = "0" + s
    data = bytes.fromhex(s)
    if len(data) < expected_len:
        data = data.ljust(expected_len, b"\x00")
    elif len(data) > expected_len:
        data = data[:expected_len]
    return data

def derive_key_nonce_from_plaintext(plaintext: str):
    """
    Derive a 128-bit key and 64-bit nonce from the UTF-8 bytes
    of the plaintext using SHA-256.

    key   = first 16 bytes  of SHA-256(plaintext)
    nonce = next  8 bytes   of SHA-256(plaintext)
    """
    if not plaintext:
        plaintext = "default-plaintext-for-edulight-64"
    data = plaintext.encode("utf-8")
    digest = hashlib.sha256(data).digest()  # 32 bytes

    key = digest[:16]        # 128-bit key
    nonce = digest[16:24]    # 64-bit nonce
    return key, nonce

# ------------- Gradio callback functions ------------- #

def ui_encrypt(plaintext: str, key_hex: str, nonce_hex: str, rounds: int):
    try:
        # If key or nonce is empty -> derive them from plaintext
        if (not key_hex.strip()) or (not nonce_hex.strip()):
            key, nonce = derive_key_nonce_from_plaintext(plaintext)
            key_hex_out = key.hex()
            nonce_hex_out = nonce.hex()
        else:
            key = _hex_normalize(key_hex, 16)
            nonce = _hex_normalize(nonce_hex, 8)
            key_hex_out = key_hex.strip().replace(" ", "").lower()
            nonce_hex_out = nonce_hex.strip().replace(" ", "").lower()

        data = plaintext.encode("utf-8")
        cipher = encrypt_ctr(data, key, nonce, rounds=rounds)
        ciphertext_hex = cipher.hex()

        # Return ciphertext and also (possibly updated) key/nonce hex
        return ciphertext_hex, key_hex_out, nonce_hex_out

    except Exception as e:
        return f"Error: {e}", key_hex, nonce_hex

def ui_decrypt(cipher_hex: str, key_hex: str, nonce_hex: str, rounds: int):
    try:
        key = _hex_normalize(key_hex, 16)
        nonce = _hex_normalize(nonce_hex, 8)
        cipher = bytes.fromhex(cipher_hex.strip().replace(" ", ""))
        plain = decrypt_ctr(cipher, key, nonce, rounds=rounds)
        return plain.decode("utf-8", errors="replace")
    except Exception as e:
        return f"Error: {e}"

def ui_run_tests(rounds: int):
    try:
        aval_pt = avalanche_plaintext(rounds=rounds, trials=200)
        aval_key = avalanche_key(rounds=rounds, trials=200)
        msg = (
            f"EduLight-64 Strength Tests (Rounds = {rounds})\n\n"
            f"- Average bit flips (plaintext avalanche): {aval_pt:.2f} / 64\n"
            f"- Average bit flips (key avalanche)      : {aval_key:.2f} / 64\n"
            f"\nNote: Ideal random behavior ~32 bits for a 64-bit block.\n"
        )
        return msg
    except Exception as e:
        return f"Error: {e}"

# ------------- Build Gradio UI ------------- #

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # EduLight-64: Lightweight Custom Cipher (Demo UI)

        **WARNING:** This cipher is for information security education and experimentation only.
        Do **not** use it to protect real or sensitive data.

        - Block size: **64 bits**
        - Key size: **128 bits**
        - Rounds: configurable (default 16)
        - Mode: **CTR** (stream-like encryption)

        If you leave the **Key** or **Nonce** fields empty, they are **automatically derived**
        from the UTF-8 bytes of your plaintext using SHA-256.
        """
    )

    with gr.Tab("Encrypt"):
        with gr.Row():
            with gr.Column():
                plaintext = gr.Textbox(
                    label="Plaintext (UTF-8)",
                    placeholder="Type the message you want to encrypt..."
                )
                key_enc = gr.Textbox(
                    label="Key (hex, 128-bit = 32 hex chars) — leave empty to auto-generate",
                    value=""
                )
                nonce_enc = gr.Textbox(
                    label="Nonce / IV (hex, 64-bit = 16 hex chars) — leave empty to auto-generate",
                    value=""
                )
                rounds_enc = gr.Slider(
                    label="Rounds",
                    minimum=8,
                    maximum=24,
                    value=16,
                    step=1
                )
                btn_enc = gr.Button("Encrypt")
            with gr.Column():
                cipher_out = gr.Textbox(
                    label="Ciphertext (hex)",
                    interactive=False,
                    lines=5
                )

        btn_enc.click(
            fn=ui_encrypt,
            inputs=[plaintext, key_enc, nonce_enc, rounds_enc],
            outputs=[cipher_out, key_enc, nonce_enc]
        )

    with gr.Tab("Decrypt"):
        with gr.Row():
            with gr.Column():
                cipher_in = gr.Textbox(
                    label="Ciphertext (hex)",
                    placeholder="Paste hex ciphertext here..."
                )
                key_dec = gr.Textbox(
                    label="Key (hex, 128-bit = 32 hex chars)",
                    value=""
                )
                nonce_dec = gr.Textbox(
                    label="Nonce / IV (hex, 64-bit = 16 hex chars)",
                    value=""
                )
                rounds_dec = gr.Slider(
                    label="Rounds",
                    minimum=8,
                    maximum=24,
                    value=16,
                    step=1
                )
                btn_dec = gr.Button("Decrypt")
            with gr.Column():
                plain_out = gr.Textbox(
                    label="Decrypted Plaintext (UTF-8)",
                    interactive=False,
                    lines=5
                )

        btn_dec.click(
            fn=ui_decrypt,
            inputs=[cipher_in, key_dec, nonce_dec, rounds_dec],
            outputs=plain_out
        )

    with gr.Tab("Strength Tests"):
        rounds_test = gr.Slider(
            label="Rounds",
            minimum=8,
            maximum=24,
            value=16,
            step=1
        )
        btn_test = gr.Button("Run Avalanche Tests")
        test_out = gr.Textbox(
            label="Results",
            lines=8,
            interactive=False
        )

        btn_test.click(
            fn=ui_run_tests,
            inputs=[rounds_test],
            outputs=test_out
        )

demo.launch()


c:\Users\os    05\Desktop\IS-FINAL PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\os    05\AppData\Local\Temp\ipykernel_8260\3748463798.py:93: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
